# Project Group 01 - NLP

**Group Members Name:**  

1.   Abdul Ghafoor 26110202
2.   Ahmed Awan 26110272


1.   Areen Sohail 26110177
2.   Rafia Imran 26110360


**Date:** 28 April 2026

Answers where needed are in comments followed by '#' comments are also followed by '#'

**Please Note that we have used groq and paid Chatgpt model their apis will expire on 25 MAY 2026 so after that it may give an error so update the keys accordingly**

# Installing Libraries

In [ ]:
!pip install textstat

In [ ]:
!pip install nrclex

In [ ]:
!pip install groq

In [ ]:
!pip install emoji

In [ ]:
!pip install gensim

In [ ]:
!pip install ydata-profiling

In [ ]:
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import pandas as pd
import numpy as np
import seaborn as sns
import warnings
import re
import string
import nltk
import missingno as msno
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.tokenize import RegexpTokenizer
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import wordnet
from gensim.utils import simple_preprocess
from pprint import pprint
from wordcloud import WordCloud, STOPWORDS
from matplotlib import pyplot as plt
from collections import Counter
from nltk.tag import pos_tag
import emoji
import networkx as nx
import textstat
from openai import OpenAI

from sklearn.model_selection import train_test_split
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report ,confusion_matrix
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.metrics import ConfusionMatrixDisplay
from nrclex import NRCLex
from sklearn.decomposition import NMF
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

import os
import google.generativeai as genai
from google.colab import userdata
from IPython.display import display, Markdown
from google.colab import userdata
from groq import Groq

from collections import Counter
from nltk.corpus import stopwords
from nltk.util import bigrams, trigrams
from nltk.probability import FreqDist
from sklearn.metrics import precision_recall_fscore_support

warnings.filterwarnings('ignore')

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')
nltk.download('vader_lexicon')
nltk.download('averaged_perceptron_tagger_eng')

# Loading The Dataset and Initial Feel of the Data

In [ ]:
df = pd.read_excel('/content/Diabetes Continuous Glucose Monitoring – Data.xlsx')

In [ ]:
df

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

# Phase 1: Data Preparation & Preprocessing

## Data Exploration: Finding number of Missing values, Attribute Types, Content in the data.

In [ ]:
# Getting an understanding of the rows and columns of data.

df.shape

In [ ]:
df.describe()

In [ ]:
# Checking how many unique values are in key categorical columns

for col in ['Source Type', 'Post Type', 'Sentiment', 'Is Paid', 'Media Type', 'Author Gender']:
    print(f'{col}: {df[col].nunique()} unique values')
    print(df[col].value_counts().head(5))
    print()

# Source Type tells us where the posts come from — Forums, Twitter, Instagram, Blogs
# Sentiment was pre-labeled by the data provider as Positives / Negatives / Neutrals / Mixed

In [ ]:
# Summary statistics for numeric engagement columns

engagement_cols = ['Total Engagements', 'Post Comments', 'Post Likes', 'Post Shares', 'Post Views', 'Richness']
df[engagement_cols].describe().round(2)

# Engagement is very skewed — a few posts drive disproportionately high numbers, we can also see post shares and Post viws are missing, Total engagment data, Post Comments, post likes also have alot of missing information.. will have to see if we can use it for modelling or not

In [ ]:
msno.bar(df, color='lightskyblue')

plt.title('Missing Value Profile — CGM Social Media Dataset')

plt.tight_layout()
plt.show()

# Several metadata columns like Author Gender, Location, Professions are heavily missing
# Most critically, Sound Bite Text (the raw post text we need) looks quite complete — good news

In [ ]:
missing_counts = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing_counts / len(df) * 100).round(2)
missing_summary = pd.DataFrame({'Missing Count': missing_counts, 'Missing %': missing_pct})

print(missing_summary[missing_summary['Missing Count'] > 0])

# The entire suite of 'LexisNexis' features (Source Quality, Company, Person, Subjects, etc.), 'Ratings and Scores', 'Post Views', 'Tags', and '@Mention Media Tags' are entirely empty (37,844 rows, 100% missing). There is no data to impute here. These are structural artifacts from the data provider and must be dropped immediately.
# As expected with social media (e.g., Reddit/anonymized forums), users do not expose granular data. 'Author Location' (City 1 & 2, State/Province 1 & 2, Country 2) are 99%+ missing. Similarly, 'Professions' (99.4%) and 'Interests' (99.3%) are practically non-existent. Engagement metrics ('Post Comments', 'Post Likes', 'Total Engagements', 'Post Dislikes', 'Post Shares') are missing for over 98% of the corpus. We cannot perform any meaningful engagement analysis or demographic clustering with <2% of the data. We will Drop them.
# Negative Objects (87.9% missing) and 'Positive Objects' (76.7% missing) are far too sparse. Relying on these pre-computed, third-party tags would introduce massive bias into the dataset. We will handle sentiment and object relation natively in our own pipeline rather than trusting highly incomplete metadata.
# Followers/Subscribers' (18.5%), 'Source Name' (10.9%), 'Post Type' (7%), 'Title' (4%), and 'Author Name' (2.5%) have acceptable missing rates. We will retain these. Missing categorical fields will be imputed with an 'Unknown' flag, and numericals (like Followers) can be imputed using the median to avoid skewing our baseline distributions.

In [ ]:
# Colomns with no missing values

print(missing_summary[missing_summary['Missing Count'] == 0])

In [ ]:
# Getting a more feel of specific filled colomns to get a more deeper understanding of what is happening in the data set

df['Media Type'].unique()

In [ ]:
# Looking over the values of Sentiment

df['Sentiment'].unique()

In [ ]:
df['Richness'].unique()

In [ ]:
df['Richness'].mean()

In [ ]:
df['Sound Bite Text'].head()

## Dropping Columns

In [ ]:
# Dropping rows where Sound Bite Text is missing since that is the core of our analysis
df = df.dropna(subset=['Sound Bite Text']).reset_index(drop=True)

print('Rows after dropping missing text:', len(df))

In [ ]:
# Dropping columns that are 100% null — they carry zero information
cols_to_drop = [col for col in df.columns if df[col].isnull().mean() == 1.0]

df.drop(columns=cols_to_drop, inplace=True)

print(f'Dropped {len(cols_to_drop)} fully-null columns')
print(f'Remaining shape: {df.shape}')

In [ ]:
# Also dropping columns that are >90% null or are admin/tracking only
high_null  = [col for col in df.columns if df[col].isnull().mean() > 0.90]
admin_cols = ['Media Link', 'Author URL', 'Author ID']

df.drop(columns=high_null + [c for c in admin_cols if c in df.columns], inplace=True)

print(f'Working shape: {df.shape}')

# Keeping: Sound Bite Text, Sentiment, Source Type, Post Type, Published Date, Author info, Engagement metrics

In [ ]:
df

In [ ]:
df['Source Type'].unique()

In [ ]:
print(df.isnull().sum())

# Sound Bite Text has zero missing values, so all rows can be used for NLP

## Converting Data Types into floats/Int

In [ ]:
df.info()

In [ ]:
# Converting Author Reddit Karma and Reddit Score to Float

df['Author Reddit Karma'] = df['Author Reddit Karma'].replace('-', np.nan, regex=False)
df['Reddit Score'] = df['Reddit Score'].replace('-', np.nan, regex=False)

# 2. Strip out any commas from strings
df['Author Reddit Karma'] = df['Author Reddit Karma'].astype(str).str.replace(',', '', regex=False)
df['Reddit Score'] = df['Reddit Score'].astype(str).str.replace(',', '', regex=False)

In [ ]:
df['Author Reddit Karma'] = pd.to_numeric(df['Author Reddit Karma'], errors='coerce')
df['Reddit Score'] = pd.to_numeric(df['Reddit Score'], errors='coerce')

In [ ]:
print(df[['Author Reddit Karma', 'Reddit Score']].info())

# After Cleaning We can see that Author Reddit Karma has almost 10,000 missing values.. imputing the mdeian value in it could prove problametic as we can see that there are people who have posted via different platforms so these missing values are those and shall be treated as such
# Similary for Reddit score above logic

## Handling Missing values by Imputations

In [ ]:
followers_median = df['Followers/Daily Unique Visitors/Subscribers'].median()

In [ ]:
print(followers_median)

In [ ]:
# The missing values of Author Reddit Karma and Score will be 0 as there are other platforms as well and comments are there as well so those dont have these metrics and cant remove these values
df['Author Reddit Karma'] = df['Author Reddit Karma'].fillna(0)
df['Reddit Score'] = df['Reddit Score'].fillna(0)

# Using median value to caclulate the missing followers
df['Followers/Daily Unique Visitors/Subscribers'] = df['Followers/Daily Unique Visitors/Subscribers'].fillna(followers_median)

## Text Cleaning

In [ ]:
# Lowering Text

df['text_clean'] = df['Sound Bite Text'].str.lower()
df['text_clean'].head(3)

In [ ]:
# Social media text is messy — has URLs, @mentions, #hashtags, emojis, reddit formatting
# We need to strip all of that out before any NLP

def clean_social_media_text(text):
    if not isinstance(text, str):
        return ''

    text = emoji.demojize(text, delimiters=(" ", " "))     # Getting information from the emojis by converting it into text

    text = re.sub(r'http\S+|www\.\S+', '', text)           # remove URLs
    text = re.sub(r'@\w+', '', text)                       # remove @mentions
    text = re.sub(r'#(\w+)', r'\1', text)                  # strip # but keep the word
    text = re.sub(r'\\n|\n|\r|\t', ' ', text)              # fix newlines and tabs
    text = re.sub(r'[^\w\s\-\.]', ' ', text)
    text = re.sub(r'\bxx+\b', '', text)                    # remove xxxx placeholders
    text = re.sub(r'\s+', ' ', text).strip()               # collapse multiple spaces

    return text.lower()

df['text_clean'] = df['text_clean'].apply(clean_social_media_text)


print(df[['Sound Bite Text', 'text_clean']].head(3))

In [ ]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    if not isinstance(text, str):
        return ''
    tokens = word_tokenize(text)
    filtered = [w for w in tokens if w.isalpha() and w not in stop_words]
    return ' '.join(filtered)

df['text_clean'] = df['text_clean'].apply(remove_stopwords)
df['text_clean'].head(3)

In [ ]:
# Checking the top 40 most common words to spot any non-informative high freq words
all_words = ' '.join(df['text_clean'].dropna()).split()
word_freq = Counter(all_words)
top_40 = word_freq.most_common(40)

top40_df = pd.DataFrame(top_40, columns=['Word', 'Frequency'])
print('Top 40 Most Frequent Words:')
print(top40_df.to_string())

# Words like 'cgm', 'glucose', 'sensor', 'dexcom', 'blood', 'diabetes', 'sugar' dominate
# These ARE meaningful for our analysis unlike generic filler words
# However generic English words like 'also', 'would', 'use' were removed by iterative process


In [ ]:
# Removing domain-generic filler words that appear everywhere but say nothing about opinions
# These were added by rerunning the top most frequent words again and again
# These are words that are extremely frequent but carry no signal about patient experience
domain_fillers = [
    'also', 'would', 'use', 'using', 'used', 'like', 'one', 'get', 'got',
    'know', 'think', 'going', 'go', 'really', 'even', 'still', 'want',
    'time', 'way', 'us', 'well', 'could', 'make', 'much', 'since', 'see',
    'need', 'back', 'take', 'im', 'ive', 'dont', 'its', 'thats', 'ive',
    'said', 'say', 'thing', 'every', 'day', 'year', 'month', 'week',
    'new', 'right', 'put', 'good', 'old', 'lot', 'little', 'ever',
    'never', 'always', 'maybe', 'first', 'last', 'come', 'came','getting',
    'sure', 'keep', 'feel', 'people', 'times', 'days', 'years', 'work', 'help'
]

def remove_filler_words(text):
    words = text.split()
    words = [w for w in words if w.lower() not in domain_fillers]
    return ' '.join(words)

df['text_clean'] = df['text_clean'].apply(remove_filler_words)

print('Sample cleaned text:')
print(df['text_clean'].iloc[0])

## Stemming and Lemmatization


In [ ]:
# Defining

stemmer    = PorterStemmer()
lemmatizer = WordNetLemmatizer()

In [ ]:
# Creating the function to get stems and lemmas

def get_stems_and_lemmas(clean_text):
    # Handle missing or empty values gracefully
    if not isinstance(clean_text, str):
        return [], []

    # Split the already-clean string into a list of words
    tokens = clean_text.split()

    # Generate stems and lemmas directly
    stems = [stemmer.stem(word) for word in tokens]
    lemmas = [lemmatizer.lemmatize(word) for word in tokens]

    return stems, lemmas

In [ ]:
df['stems'], df['lemmas'] = zip(*df['text_clean'].apply(get_stems_and_lemmas))

df[['text_clean', 'stems', 'lemmas']].head()

In [ ]:
# Most common lemmas
common_lemmas = df['lemmas'].explode().value_counts().head(20)

print('Most Common Lemmas:')
print(common_lemmas)

# words like dexcom, cgm, pump, insulin, glucose, sensor, sugar, and blood are very common — typical for discussions around continuous glucose monitoring and diabetes management

In [ ]:
# Vizualization for better understanding

# Most common lemmas — these tell us what CGM patients talk about at the root word level
common_lemmas = df['lemmas'].explode().value_counts().head(25)

plt.figure(figsize=(10, 7))
plt.barh(common_lemmas.index[::-1], common_lemmas.values[::-1],
         color='lightskyblue', edgecolor='deepskyblue')
plt.xlabel('Frequency')
plt.ylabel('Lemma')
plt.title('Top 25 Most Common Lemmas in CGM Social Media Posts')
plt.tight_layout()
plt.show()

# sensor, glucose, blood, dexcom, pump, insulin, libre, reading, level, accuracy are the top lemmas
# These confirm the dataset is rich with CGM-specific terminology — device names, medical concepts, usage terms

In [ ]:
# Converting lemma lists to strings for downstream modeling
df['lemmas_string'] = df['lemmas'].apply(lambda x: ' '.join(x) if isinstance(x, list) else '')

# Also creating a stems string just in case we need it
df['stems_string'] = df['stems'].apply(lambda x: ' '.join(x) if isinstance(x, list) else '')

print('Sample lemma string:')
print(df['lemmas_string'].iloc[0])

print('')

print('Sample stem string:')
print(df['stems_string'].iloc[0])

## POS TAGGING

Part-of-Speech (POS) Tagging is the process of algorithmically reading a sentence and labeling every single word with its grammatical category based on its definition and its context.


---



In [ ]:
# So Basically for this function what we have done is that it first calculates the POS Tags for each word in the sentence based on the context etc, and we are doing this by updating the global counter which stores the number of times each verb/etc is coming so we can make a barchart out of it then it is creating a new column that has only Noun and Adjective so we can have a detailed analysis of it and we can cut out only noise in the data set for further analysis

# Initialize the global counter for the bar chart
global_pos_counts = Counter()

def process_pos_and_extract_pairs(text):
    if not isinstance(text, str) or not text.strip():
        return [] # Return an empty list for the new column if text is null

    # Perform the expensive tokenization/tagging exactly ONCE
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)

    # Update the global tag counter
    tags_only = [tag for word, tag in tagged]
    global_pos_counts.update(tags_only)

    # Extract the Noun-Adjective pain points
    pain_points = []
    for i in range(len(tagged) - 1):
        word1, tag1 = tagged[i]
        word2, tag2 = tagged[i+1]

        # Rule: Noun (NN) followed by Adjective (JJ)
        if tag1.startswith('JJ') and tag2.startswith('NN'):
            pain_points.append(f"{word1}_{word2}")

    # Return the extracted pairs to be saved in the dataframe
    return pain_points

In [ ]:
print("Executing unified POS pipeline across 37k rows... please wait.")

# We apply the function and instantly capture the returned list into our new column

df['Extracted_Features'] = df['text_clean'].apply(process_pos_and_extract_pairs)

In [ ]:
print(df[['text_clean', 'Extracted_Features']].head(5))

In [ ]:
top_pos = pd.DataFrame(global_pos_counts.most_common(15), columns=['POS Tag', 'Frequency'])

plt.figure(figsize=(10, 5))
plt.bar(top_pos['POS Tag'], top_pos['Frequency'], color='lightsalmon', edgecolor='salmon')
plt.xlabel('POS Tag', fontweight='bold')
plt.ylabel('Frequency', fontweight='bold')
plt.title('Top 15 POS Tag Frequencies in CGM Posts (Full 37k Corpus)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print(top_pos.to_string())

Analysis of POS to be Added in Presentation and report in much detail - AJ should work on it

1. ENTITY-DRIVEN CONVERSATION (NN, NNS): Nouns dominate the corpus (>500k occurrences).
This indicates the discourse is highly focused on tangible product components (hardware,
apps, adhesives) and systemic entities (insurance, doctors), making this dataset
ideal for extracting specific feature-level unmet needs.

2. HIGHLY EVALUATIVE SENTIMENT (JJ, JJR, JJS): Over 216k adjectives indicate a heavily
opinionated dataset. The presence of comparative (JJR) and superlative (JJS) tagssuggests patients are frequently benchmarking products against one another (e.g., "better", "worst"), which is critical for competitive market analysis.

3. ACTION & FRICTION (VBG, VBD): The wide distribution of verb tenses highlights the operational burden of diabetes. The prominence of VBG (present participle, often ending in -ing) points to ongoing mechanical/digital interactions or failures (e.g., "syncing", "bleeding", "failing"), while past tense verbs (VBD) indicate users are sharing specific past incidents and case studies.

4. EMOTIONAL INTENSITY (RB, RBR): Over 78k adverbs act as amplifiers ("extremely", "never", "always"), signifying high emotional friction and distress in the patient journey.

## Column Engineering for Modelling

In [ ]:
# For Finding the Brand_names since that is missing in the data set


def extract_brand_target(text):
    if not isinstance(text, str):
        return 'Other'

    # Standardize for matching
    text_lower = text.lower()

    # Define our competitive intelligence keywords
    dexcom_terms = ['dexcom', 'g6', 'g7', 'g5', 'dex']
    libre_terms = ['libre', 'freestyle', 'libre 2', 'libre 3', 'libre2', 'libre3']

    # Check for presence of keywords
    is_dexcom = any(term in text_lower for term in dexcom_terms)
    is_libre = any(term in text_lower for term in libre_terms)

    # Logic routing
    if is_dexcom and is_libre:
        return 'Both (Comparison)'
    elif is_dexcom:
        return 'Dexcom'
    elif is_libre:
        return 'Freestyle Libre'
    else:
        return 'Other'

In [ ]:
df['Brand_Name'] = df['Sound Bite Text'].apply(extract_brand_target)

print(df[['Sound Bite Text', 'Brand_Name']].head(10))

### For Phase 4: DataFrame Split

In [ ]:
# Creating brand-specific dataframes for focused analysis
df_dexcom = df[df['Brand_Name'].isin(['Dexcom', 'Both'])].copy()
df_libre  = df[df['Brand_Name'].isin(['Freestyle Libre', 'Both'])].copy()

print(f'Dexcom posts (including Both): {len(df_dexcom)}')
print(f'Freestyle Libre posts (including Both): {len(df_libre)}')

In [ ]:
df

# Phase 2: Exploratory Data Analysis (EDA)

## Network Graph of Pain Points - POS TAGGING

In [ ]:
# Basically this graph is showing the pain points of customers for detailed analysis and EDA, of what could be the underlying reasons some customers could be feeling the way they are feeling

# Flattening extracted Adjective-Noun pairs into a single list
all_pairs = [pair for sublist in df['Extracted_Features'].dropna() for pair in sublist]

# Counting the frequencies and take the top 100 most common relationships
pair_counts = Counter(all_pairs)
top_pairs = pair_counts.most_common(100)

# 3. Initializing the Network Graph
G = nx.Graph()

# Populating the nodes and edges
for pair, count in top_pairs:
    # Split the "Noun_Adjective" string back into two separate words
    try:
        noun, adj = pair.split('_')
        # Add the connection to the graph, using the frequency as the weight/thickness
        G.add_edge(noun, adj, weight=count)
    except ValueError:
        continue # Skip any malformed pairs

# visualization
plt.figure(figsize=(14, 10))

# Using a spring layout to push highly connected nodes to the center
pos = nx.spring_layout(G, k=0.5, iterations=40)

# Drawing the nodes (Hardware = center nodes, Adjectives = outer nodes)
nx.draw_networkx_nodes(G, pos, node_size=10000, node_color='skyblue', alpha=0.6)

# Drawing the edges (Thicker lines = more frequent complaints)
edges = G.edges(data=True)
weights = [edge[2]['weight'] / max([e[2]['weight'] for e in edges]) * 5 for edge in edges]
nx.draw_networkx_edges(G, pos, width=weights, edge_color='gray', alpha=0.5)

# Adding the labels
nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold')

plt.title("Patient Friction Web: Hardware vs. Experience", fontsize=16, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

## Sentiment Comparison of two Brands

In [ ]:
# Side-by-side comparison of sentiment category breakdown by brand

dexcom_sent = df_dexcom['Sentiment'].value_counts(normalize=True) * 100
libre_sent  = df_libre['Sentiment'].value_counts(normalize=True) * 100

sent_comparison = pd.DataFrame({'Dexcom': dexcom_sent, 'Freestyle Libre': libre_sent}).fillna(0)

sent_comparison.plot(kind='bar', figsize=(10, 6), color=['lightskyblue', 'lightsalmon'],
                     edgecolor='grey')

plt.xlabel('Sentiment Category')
plt.ylabel('Percentage of Posts (%)')
plt.title('Sentiment Category Breakdown: Dexcom vs Freestyle Libre')
plt.xticks(rotation=0)
plt.legend()
plt.tight_layout()
plt.show()

print(sent_comparison.round(2))

# This directly shows which brand has a higher % of positive vs negative discussions
# Freestyle has fewer % negative posts, it signals better patient satisfaction

In [ ]:
print("Final Dataset Shape:", df.shape)

summary = pd.DataFrame({
    'Column': df.columns,
    'Data Type': df.dtypes.values,
    'Missing %': (df.isnull().mean() * 100).round(2).values
})

summary.sort_values(by='Missing %', ascending=False).head(15)

In [ ]:
print(df.columns)

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(data=df, x='Source Type', order=df['Source Type'].value_counts().index, palette='Blues')

plt.title('Distribution of Posts by Source Type')
plt.xlabel('Source Type')
plt.ylabel('Count')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print(df['Source Type'].value_counts(normalize=True).round(3))

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='Sentiment', order=df['Sentiment'].value_counts().index, palette='coolwarm')

plt.title('Overall Sentiment Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# Percentage view
sentiment_pct = df['Sentiment'].value_counts(normalize=True) * 100
print(sentiment_pct.round(2))

In [ ]:
platform_sentiment = pd.crosstab(df['Source Type'], df['Sentiment'], normalize='index') * 100

platform_sentiment.plot(kind='bar', stacked=True, figsize=(10,6), colormap='coolwarm')

plt.title('Sentiment Distribution Across Platforms')
plt.ylabel('Percentage (%)')
plt.xlabel('Source Type')
plt.legend(title='Sentiment')
plt.tight_layout()
plt.show()

In [ ]:
df = df.rename(columns={
    "Published Date (GMT-04:00) New York": "Published Date"
})

df['Published Date'] = pd.to_datetime(df['Published Date'], errors='coerce')

df['Year-Month'] = df['Published Date'].dt.to_period('M')

monthly_posts = df['Year-Month'].value_counts().sort_index()

monthly_posts.plot(figsize=(10,5))

plt.title('Posting Trend Over Time')
plt.xlabel('Time')
plt.ylabel('Number of Posts')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
platform_time = df.groupby(
    [df['Published Date'].dt.to_period('M'), 'Source Type']
).size().unstack().fillna(0)

platform_time.plot(figsize=(10,6))

plt.title('Platform-wise Posting Trends Over Time')
plt.xlabel('Time')
plt.ylabel('Post Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
df['text_length'] = df['text_clean'].apply(lambda x: len(x.split()))

plt.figure(figsize=(8,5))
sns.histplot(df['text_length'], bins=50, kde=True)

plt.title('Distribution of Text Length')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x='Sentiment', y='text_length', palette='coolwarm')

plt.title('Text Length by Sentiment')
plt.xlabel('Sentiment')
plt.ylabel('Word Count')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='Brand_Name', order=df['Brand_Name'].value_counts().index)

plt.title('Brand Mentions Distribution')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print(df['Brand_Name'].value_counts(normalize=True).round(3))

In [ ]:
brand_sent = pd.crosstab(df['Brand_Name'], df['Sentiment'], normalize='index') * 100

brand_sent.plot(kind='bar', stacked=True, figsize=(10,6), colormap='coolwarm')

plt.title('Brand vs Sentiment Distribution')
plt.ylabel('Percentage (%)')
plt.xlabel('Brand')
plt.legend(title='Sentiment')
plt.tight_layout()
plt.show()

print(brand_sent.round(2))

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(
    df['Followers/Daily Unique Visitors/Subscribers'],
    bins=50,
    kde=True
)

plt.title('Distribution of Audience Size')
plt.xlabel('Followers / Subscribers')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
if 'Post Type' in df.columns:
    plt.figure(figsize=(7,5))
    sns.countplot(data=df, x='Post Type', order=df['Post Type'].value_counts().index)

    plt.title('Distribution of Post Types')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

    print(df['Post Type'].value_counts(normalize=True).round(3))

In [ ]:
if 'Media Type' in df.columns:
    media_dist = df['Media Type'].value_counts(normalize=True) * 100

    plt.figure(figsize=(6,4))
    media_dist.plot(kind='bar')

    plt.title('Media Type Distribution (%)')
    plt.ylabel('Percentage')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

    print(media_dist.round(2))

In [ ]:
tokens = df['text_clean'].str.split()

# Bigrams
bigram_list = []
for row in tokens:
    bigram_list.extend(list(bigrams(row)))

bigram_freq = Counter(bigram_list).most_common(20)

bigram_df = pd.DataFrame(bigram_freq, columns=['Bigram', 'Frequency'])

plt.figure(figsize=(10,6))
sns.barplot(x='Frequency', y=bigram_df['Bigram'].astype(str), data=bigram_df)

plt.title('Top 20 Bigrams')
plt.tight_layout()
plt.show()

In [ ]:
text_corpus = ' '.join(df['text_clean'])

wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text_corpus)

plt.figure(figsize=(10,5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of CGM Discussions')
plt.show()

# Phase 3: General CGM Analysis (Topic Modeling & Needs Discovery)

## Setup

In [ ]:
df_topic = df.copy()
print("Dataset shape:", df_topic.shape)
print("\nColumns:")

print(df_topic.columns.tolist())
df_topic.head()

In [ ]:
# Selecting the best available text column for analysis

possible_text_columns = [
    'lemmas_string',
    'text_clean',
    'cleaned_text',
    'Sound Bite Text',
    'Text',
    'Post',
    'Body'
]

text_col = None

for col in possible_text_columns:
    if col in df_topic.columns:
        text_col = col
        break

if text_col is None:
    raise ValueError("No suitable text column found. Please check the dataset columns.")

print("Text column selected:", text_col)

## Topic Modeling Preparation

Topic modeling is used to identify the main themes in the CGM discussion. Since social media posts and patient comments are usually short and noisy, TF-IDF is used to highlight important words and phrases while reducing the impact of very common words.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

topic_vectorizer = TfidfVectorizer(
    max_df=0.85,
    min_df=20,
    max_features=5000,
    ngram_range=(1, 2),
    stop_words='english'
)

X_topics = topic_vectorizer.fit_transform(df_topic[text_col])

print("TF-IDF matrix shape:", X_topics.shape)

## Topic Modeling Using NMF

Non-Negative Matrix Factorization is used for topic modeling because it works well with TF-IDF features and usually produces interpretable topics for short text data.

Each post is assigned to the topic for which it has the strongest score.

In [ ]:
num_topics = 8

nmf_model = NMF(
    n_components=num_topics,
    random_state=42,
    init='nndsvda',
    max_iter=500
)

topic_matrix = nmf_model.fit_transform(X_topics)

# Assign dominant topic to each post
df_topic['Topic_ID'] = topic_matrix.argmax(axis=1)
df_topic['Topic_Strength'] = topic_matrix.max(axis=1)

df_topic[[text_col, 'Topic_ID', 'Topic_Strength']].head()

In [ ]:
feature_names = topic_vectorizer.get_feature_names_out()

def display_topics(model, feature_names, num_words=15):
    topic_word_dict = {}

    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[:-num_words - 1:-1]
        top_words = [feature_names[i] for i in top_indices]
        topic_word_dict[topic_idx] = top_words

        print(f"\nTopic {topic_idx}:")
        print(", ".join(top_words))

    return topic_word_dict

topic_words = display_topics(nmf_model, feature_names, num_words=15)

## Topic Interpretation

The topics generated by the model are interpreted manually based on the highest-weighted words in each topic.

The labels below convert statistical topic outputs into meaningful CGM discussion themes.

In [ ]:
# Topic labels interpreted from the dataset's top words

topic_labels = {
    0: "General CGM Use & Daily Management",
    1: "Glucose Readings & Accuracy",
    2: "Dexcom Discussion",
    3: "Insulin, Pump & Treatment Management",
    4: "App, Alerts & Connectivity",
    5: "Sensor Wear, Replacement & Reliability",
    6: "Lifestyle, Food & Exercise",
    7: "Insurance, Cost & Access"
}

df_topic['Topic_Label'] = df_topic['Topic_ID'].map(topic_labels)

df_topic[[text_col, 'Topic_ID', 'Topic_Label']].head()

In [ ]:
topic_distribution = (
    df_topic['Topic_Label']
    .value_counts()
    .reset_index()
)

topic_distribution.columns = ['Topic', 'Post Count']

topic_distribution['Percentage'] = (
    topic_distribution['Post Count'] / topic_distribution['Post Count'].sum() * 100
).round(2)

topic_distribution

In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    data=topic_distribution,
    x='Post Count',
    y='Topic'
)

plt.title('Dominant CGM Conversation Topics')
plt.xlabel('Number of Posts')
plt.ylabel('Topic')
plt.tight_layout()
plt.show()

## Sentiment by Topic

After assigning topics, sentiment is compared across each topic. This helps identify which CGM themes are associated with satisfaction and which themes are associated with frustration or unmet needs.

In [ ]:
# Selecting sentiment column

possible_sentiment_columns = [
    'Sentiment',
    'sentiment',
    'Sentiment Label',
    'sentiment_label'
]

sentiment_col = None

for col in possible_sentiment_columns:
    if col in df_topic.columns:
        sentiment_col = col
        break

if sentiment_col is None:
    raise ValueError("No sentiment column found. Please check the dataset columns.")

print("Sentiment column selected:", sentiment_col)
print(df_topic[sentiment_col].value_counts())

In [ ]:
topic_sentiment = pd.crosstab(
    df_topic['Topic_Label'],
    df_topic[sentiment_col],
    normalize='index'
) * 100

topic_sentiment = topic_sentiment.round(2)

topic_sentiment

In [ ]:
topic_sentiment.plot(
    kind='bar',
    stacked=True,
    figsize=(14, 7),
    edgecolor='black'
)

plt.title('Sentiment Breakdown by CGM Topic')
plt.xlabel('Topic')
plt.ylabel('Percentage of Posts')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Sentiment')
plt.tight_layout()
plt.show()

## High-Friction Topic Analysis

High-friction topics are topics with a larger share of negative or mixed sentiment. These areas are especially important because they reveal where patients experience confusion, dissatisfaction, or barriers in their CGM journey.

In [ ]:
# Standardizing sentiment labels for friction calculation
available_sentiments = topic_sentiment.columns.tolist()
print("Available sentiment labels:", available_sentiments)

# Common negative/mixed labels
friction_labels = []

for label in available_sentiments:
    label_lower = str(label).lower()
    if 'negative' in label_lower or 'negative' == label_lower or label_lower == 'negatives':
        friction_labels.append(label)
    elif 'mixed' in label_lower:
        friction_labels.append(label)

print("Friction labels used:", friction_labels)

topic_friction = topic_sentiment.copy()

if len(friction_labels) > 0:
    topic_friction['Friction_Score'] = topic_friction[friction_labels].sum(axis=1)
else:
    topic_friction['Friction_Score'] = 0

topic_friction_summary = (
    topic_friction[['Friction_Score']]
    .sort_values(by='Friction_Score', ascending=False)
    .reset_index()
)

topic_friction_summary

## Needs Discovery

The next step is to identify recurring patient needs from the text.

A rule-based needs discovery framework is used. Each need category is represented by a set of keywords commonly associated with CGM-related patient concerns.

This approach complements topic modeling because it focuses specifically on practical patient needs and pain points.

In [ ]:
need_categories = {
    'Accuracy & Trust in Readings': [
        'accurate', 'accuracy', 'reading', 'readings', 'wrong', 'false',
        'off', 'calibrate', 'calibration', 'fingerstick', 'finger',
        'stick', 'difference', 'low', 'high'
    ],

    'Affordability & Insurance Access': [
        'insurance', 'cost', 'expensive', 'price', 'coverage', 'covered',
        'pay', 'copay', 'afford', 'pharmacy', 'prescription', 'refill',
        'money', 'supply', 'supplies'
    ],

    'Adhesive, Skin & Comfort': [
        'adhesive', 'stick', 'sticking', 'fall', 'falls', 'fell',
        'skin', 'rash', 'itch', 'irritation', 'comfortable', 'pain',
        'bleeding', 'arm', 'site', 'patch'
    ],

    'App, Alerts & Connectivity': [
        'app', 'alert', 'alerts', 'alarm', 'alarms', 'bluetooth',
        'signal', 'connect', 'connection', 'sync', 'notification',
        'phone', 'ios', 'android', 'receiver', 'share'
    ],

    'Sensor Reliability & Replacement': [
        'sensor', 'sensors', 'failed', 'failure', 'replace', 'replacement',
        'error', 'warmup', 'expired', 'restart', 'faulty', 'issue',
        'problem', 'problems'
    ],

    'Lifestyle Flexibility': [
        'exercise', 'workout', 'sleep', 'shower', 'swim', 'travel',
        'food', 'meal', 'carb', 'carbs', 'activity', 'running',
        'gym', 'work', 'school'
    ],

    'Ease of Use & Setup': [
        'easy', 'hard', 'difficult', 'setup', 'insert', 'insertion',
        'training', 'learn', 'manual', 'change', 'apply', 'install',
        'start', 'starting'
    ],

    'Doctor, Treatment & Diabetes Management': [
        'doctor', 'endo', 'endocrinologist', 'a1c', 'insulin',
        'pump', 'basal', 'bolus', 'dose', 'treatment', 'diabetes',
        't1d', 'type'
    ]
}

In [ ]:
def detect_needs(text):
    detected = []

    if not isinstance(text, str):
        return detected

    text_lower = text.lower()
    words = set(text_lower.split())

    for category, keywords in need_categories.items():
        if any(keyword.lower() in words for keyword in keywords):
            detected.append(category)

    return detected

df_topic['Detected_Needs'] = df_topic[text_col].apply(detect_needs)

df_topic[[text_col, 'Detected_Needs']].head()

In [ ]:
needs_exploded = df_topic.explode('Detected_Needs')

needs_exploded = needs_exploded[
    needs_exploded['Detected_Needs'].notna()
].copy()

print("Rows with detected needs:", needs_exploded.shape[0])

needs_exploded[[text_col, 'Detected_Needs', sentiment_col]].head()

## Most Common CGM Patient Needs

The frequency of each detected need category shows which patient needs appear most often in the CGM conversation.

In [ ]:
unmet_df = (
    needs_exploded['Detected_Needs']
    .value_counts()
    .reset_index()
)

unmet_df.columns = ['Unmet Need Category', 'Post Count']

unmet_df['Percentage of Need Mentions'] = (
    unmet_df['Post Count'] / unmet_df['Post Count'].sum() * 100
).round(2)

unmet_df

In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    data=unmet_df,
    x='Post Count',
    y='Unmet Need Category'
)

plt.title('Most Frequently Discussed CGM Patient Needs')
plt.xlabel('Number of Posts')
plt.ylabel('Need Category')
plt.tight_layout()
plt.show()

## Need Categories by Sentiment

Need frequency alone does not show whether a need is positive, neutral, or negative. Therefore, each need category is compared with sentiment.

A need category with a higher share of negative or mixed sentiment represents a stronger unmet need.

In [ ]:
need_sentiment = pd.crosstab(
    needs_exploded['Detected_Needs'],
    needs_exploded[sentiment_col],
    normalize='index'
) * 100

need_sentiment = need_sentiment.round(2)

need_sentiment

## Representative Posts for Each Need Category

Representative posts are extracted for each need category to make the analysis easier to interpret. These examples help show the real patient language behind each discovered need.

In [ ]:
# Selecting engagement column

possible_engagement_columns = [
    'Total Engagements',
    'Engagements',
    'Reddit Score',
    'Score',
    'Likes',
    'Comments'
]

engagement_col = None

for col in possible_engagement_columns:
    if col in needs_exploded.columns:
        engagement_col = col
        break

print("Engagement column selected:", engagement_col)

# Selecting source column
possible_source_columns = [
    'Source Type',
    'Source',
    'Platform',
    'Channel'
]

source_col = None

for col in possible_source_columns:
    if col in needs_exploded.columns:
        source_col = col
        break

print("Source column selected:", source_col)

In [ ]:
representative_need_posts = {}

for need in unmet_df['Unmet Need Category']:
    subset = needs_exploded[
        needs_exploded['Detected_Needs'] == need
    ].copy()

    if engagement_col is not None:
        subset = subset.sort_values(by=engagement_col, ascending=False)

    display_cols = [text_col, sentiment_col]

    if source_col is not None:
        display_cols.append(source_col)

    if engagement_col is not None:
        display_cols.append(engagement_col)

    representative_need_posts[need] = subset[display_cols].head(3)

for need, posts in representative_need_posts.items():
    print("\n" + "="*90)
    print("NEED CATEGORY:", need)
    print("="*90)

    for idx, row in posts.iterrows():
        print(f"\nSentiment: {row[sentiment_col]}")

        if source_col is not None:
            print(f"Source: {row[source_col]}")

        if engagement_col is not None:
            print(f"Engagement: {row[engagement_col]}")

        print(str(row[text_col])[:700])

## Comparision with LDA

In [ ]:
# LDA Topic Modeling — comparing with NMF gives methodological depth
# NMF tends to find cleaner, more interpretable topics
# LDA tends to find broader, overlapping themes
# Comparing both validates your topic structure

# LDA works better with raw counts (CountVectorizer) not TF-IDF
count_vec = CountVectorizer(max_features=2000, max_df=0.90, min_df=5)
doc_term_matrix = count_vec.fit_transform(df_topic['lemmas_string'].dropna())

lda = LatentDirichletAllocation(n_components=6, random_state=42, max_iter=15)
lda.fit(doc_term_matrix)

# Displaying top words per topic
feature_names = count_vec.get_feature_names_out()

print("LDA TOPICS:")
for i, topic in enumerate(lda.components_):
    top_words = [feature_names[j] for j in topic.argsort()[-12:]][::-1]
    print(f"Topic {i}: {', '.join(top_words)}")

# One-line comparison note for your report
print("\nNMF vs LDA Comparison:")
print("NMF: Produces sparse, parts-based topics — better for short social media text")
print("LDA: Probabilistic model — topics overlap, better reflects real-world ambiguity")

#Phase 4: Supervised & Unsupervised NLP (Product & Sentiment Analysis)

## Emotion Analysis

In [ ]:
def get_emotions(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return {}
    emotion = NRCLex(text)
    return emotion.emotion_scores   # changed from raw_emotion_scores

In [ ]:
# NRC emotion keywords — manual fallback, no library needed
nrc_emotions = {
    'anger':   ['angry', 'frustrat', 'annoyed', 'hate', 'furious', 'outrage', 'terrible'],
    'fear':    ['scared', 'afraid', 'fear', 'anxious', 'worry', 'panic', 'terrif'],
    'trust':   ['trust', 'reliable', 'confident', 'safe', 'accurate', 'depend', 'good'],
    'joy':     ['happy', 'love', 'great', 'excellent', 'amazing', 'wonderful', 'joy'],
    'sadness': ['sad', 'disappoint', 'unfortunate', 'miss', 'unhappy', 'depressed'],
    'disgust': ['disgusting', 'awful', 'horrible', 'ridiculous', 'useless', 'garbage'],
    'surprise':['surprised', 'unexpected', 'sudden', 'shocked', 'unbelievable', 'wow'],
    'anticipation': ['hope', 'expect', 'waiting', 'looking forward', 'soon', 'upgrade']
}

def get_emotions(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return {e: 0 for e in nrc_emotions}
    text_lower = text.lower()
    scores = {}
    for emotion, keywords in nrc_emotions.items():
        scores[emotion] = sum(1 for k in keywords if k in text_lower)
    return scores

# Test
print(get_emotions("I am scared and frustrated about the sensor failing"))

In [ ]:
# NRC Emotion Lexicon — 8 emotions beyond positive/negative
# Especially useful in healthcare: 'fear' and 'trust' are clinically meaningful


df['emotions'] = df['text_clean'].apply(get_emotions)

emotion_df = pd.json_normalize(df['emotions']).fillna(0)
emotion_df.index = df.index

# Average emotion profile for each brand
dex_emotions  = pd.json_normalize(df_dexcom['text_clean'].apply(get_emotions)).fillna(0).mean()
libre_emotions = pd.json_normalize(df_libre['text_clean'].apply(get_emotions)).fillna(0).mean()

emotion_compare = pd.DataFrame({
    'Dexcom': dex_emotions,
    'Freestyle Libre': libre_emotions
}).drop(index=['positive', 'negative'], errors='ignore')

emotion_compare.plot(kind='bar', figsize=(12, 6), color=['steelblue', 'salmon'], edgecolor='grey')

plt.title('Emotion Profile by Brand (NRC Lexicon)')
plt.xlabel('Emotion')
plt.ylabel('Average Raw Emotion Score')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print(emotion_compare.round(3))

# In a CGM context, high 'fear' = patients anxious about accuracy/hypoglycemia
# High 'trust' = brand loyalty signal
# High 'anger' = a complaint driver — actionable for customer service

## Vader Sentiment Analysis

In [ ]:
# Running VADER on the raw (uncleaned) text — VADER works best on natural language

analyzer = SentimentIntensityAnalyzer()

def get_vader_sentiment(text):
    if not isinstance(text, str):
        return 0.0
    scores = analyzer.polarity_scores(text)
    return scores['compound']

# Applying to full dataset
df['vader_score'] = df['Sound Bite Text'].apply(get_vader_sentiment)

def map_vader_to_scale(score):
    if score < -0.5:
        return 1
    elif score < -0.1:
        return 2
    elif score <= 0.1:
        return 3
    elif score <= 0.5:
        return 4
    else:
        return 5

df['vader_label'] = df['vader_score'].apply(map_vader_to_scale)

print('VADER label distribution:')
print(df['vader_label'].value_counts().sort_index())

In [ ]:
# Comparing VADER sentiment distribution for Dexcom vs Freestyle Libre

df_dexcom['vader_score'] = df_dexcom['Sound Bite Text'].apply(get_vader_sentiment)
df_dexcom['vader_label'] = df_dexcom['vader_score'].apply(map_vader_to_scale)

df_libre['vader_score']  = df_libre['Sound Bite Text'].apply(get_vader_sentiment)
df_libre['vader_label'] = df_libre['vader_score'].apply(map_vader_to_scale)

In [ ]:
# Making the plot sidebyside so we can have a deeper comparrison between the two axes[0] tells that the histogram of dexa will be first and vice verca

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_dexcom['vader_score'], bins=30, color='lightskyblue', edgecolor='deepskyblue')
axes[0].set_title('VADER Sentiment Distribution — Dexcom')
axes[0].set_xlabel('VADER Compound Score')
axes[0].set_ylabel('Post Count')
axes[0].axvline(df_dexcom['vader_score'].mean(), color='deepskyblue', linestyle='--', label=f"Mean: {df_dexcom['vader_score'].mean():.3f}")
axes[0].legend()

axes[1].hist(df_libre['vader_score'], bins=30, color='lightsalmon', edgecolor='salmon')
axes[1].set_title('VADER Sentiment Distribution — Freestyle Libre')
axes[1].set_xlabel('VADER Compound Score')
axes[1].set_ylabel('Post Count')
axes[1].axvline(df_libre['vader_score'].mean(), color='salmon', linestyle='--', label=f"Mean: {df_libre['vader_score'].mean():.3f}")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Dexcom  — Mean VADER score: {df_dexcom["vader_score"].mean():.4f}')
print(f'Libre   — Mean VADER score: {df_libre["vader_score"].mean():.4f}')

# Freestyle Libre shows higher and more consistent positive sentiment (mean ≈ 0.33), while Dexcom has a lower average sentiment (≈ 0.22) with a wider spread — indicating more mixed user experiences and greater variability in satisfaction

In [ ]:
# Counting values and force the index to include all 1-5 categories (crucial for side-by-side alignment)
dexcom_counts = df_dexcom['vader_label'].value_counts().reindex([1, 2, 3, 4, 5], fill_value=0)
libre_counts = df_libre['vader_label'].value_counts().reindex([1, 2, 3, 4, 5], fill_value=0)

# Setuping side-by-side subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Defining the sentiment color scale you wanted
bar_colors = ['lightsalmon', 'lightsalmon', 'lightskyblue', 'lightgreen', 'lightgreen']

# AXES [0]: DEXCOM
axes[0].bar(dexcom_counts.index, dexcom_counts.values, color=bar_colors, edgecolor='grey')
axes[0].set_title('VADER Sentiment Distribution (1-5) — Dexcom')
axes[0].set_xlabel('Sentiment Rating (1=Very Neg, 5=Very Pos)')
axes[0].set_ylabel('Post Count')
axes[0].set_xticks([1, 2, 3, 4, 5])

# AXES [1]: FREESTYLE LIBRE
axes[1].bar(libre_counts.index, libre_counts.values, color=bar_colors, edgecolor='grey')
axes[1].set_title('VADER Sentiment Distribution (1-5) — Freestyle Libre')
axes[1].set_xlabel('Sentiment Rating (1=Very Neg, 5=Very Pos)')
axes[1].set_ylabel('Post Count')
axes[1].set_xticks([1, 2, 3, 4, 5])

plt.tight_layout()
plt.show()

print(f'Dexcom  — Mean VADER compound score: {df_dexcom["vader_score"].mean():.4f}')
print(f'Libre   — Mean VADER compound score: {df_libre["vader_score"].mean():.4f}')

In [ ]:
# Flesch Reading Ease — measures how complex the writing is
# Hypothesis: frustrated users write longer, more structured complaints
# while satisfied users write short, casual praise

df['readability'] = df['Sound Bite Text'].apply(
    lambda x: textstat.flesch_reading_ease(str(x)) if isinstance(x, str) else None
)
df['word_count'] = df['text_clean'].apply(lambda x: len(str(x).split()))

# Correlation between readability, length, and sentiment
print("Correlation with VADER score:")
print(df[['readability', 'word_count', 'vader_score']].corr()['vader_score'].round(3))

plt.figure(figsize=(8, 5))
sns.scatterplot(data=df.sample(1000), x='word_count', y='vader_score',
                hue='Sentiment', alpha=0.5, palette='coolwarm')
plt.title('Post Length vs Sentiment Score')
plt.xlabel('Word Count')
plt.ylabel('VADER Score')
plt.tight_layout()
plt.show()

## Vader Agreement Analysis

In [ ]:
# VADER vs Provider Label Agreement Analysis
# The dataset comes with pre-labeled sentiment from the data provider (113 Industries)
# We independently computed our own VADER scores — how well do they agree?
# Disagreements are analytically meaningful: they reveal where automated scoring diverges from human/provider judgment, and help us understand each method's blind spots

# Step 1: Map VADER compound scores to the same 4-class labels the provider used
def vader_to_provider_label(score):
    if score >= 0.3:
        return 'Positives'
    elif score <= -0.1:
        return 'Negatives'
    elif score > 0.1 and score < 0.3:
        return 'Mixed'
    else:
        return 'Neutrals'  # -0.1 to 0.1

df['vader_provider_label'] = df['vader_score'].apply(vader_to_provider_label)

# Step 2: Overall agreement rate
agreement_mask = df['vader_provider_label'] == df['Sentiment']
agreement_rate = agreement_mask.mean() * 100

print(f"Overall Agreement Rate between VADER and Provider Labels: {agreement_rate:.1f}%")
print(f"Disagreement Rate: {100 - agreement_rate:.1f}%")
print(f"Total Posts: {len(df)}")
print(f"Agreements: {agreement_mask.sum()} | Disagreements: {(~agreement_mask).sum()}")

In [ ]:
# Step 3: Confusion matrix — where exactly do they disagree?
# Rows = what the provider labeled it
# Columns = what VADER labeled it
# The diagonal = agreement; off-diagonal = disagreement

agreement_matrix = pd.crosstab(
    df['Sentiment'],
    df['vader_provider_label'],
    rownames=['Provider Label'],
    colnames=['VADER Label'],
    normalize='index'
) * 100

# Reordering for logical flow
label_order = ['Positives', 'Mixed', 'Neutrals', 'Negatives']
agreement_matrix = agreement_matrix.reindex(
    index=[l for l in label_order if l in agreement_matrix.index],
    columns=[l for l in label_order if l in agreement_matrix.columns]
)

plt.figure(figsize=(9, 6))
sns.heatmap(
    agreement_matrix,
    annot=True,
    fmt='.1f',
    cmap='Blues',
    linewidths=0.5,
    vmin=0, vmax=100
)
plt.title('Provider Labels vs VADER Labels — Agreement Heatmap (%)\n(Row = Provider, Column = VADER — Diagonal = Agreement)', fontsize=12)
plt.tight_layout()
plt.show()

print("\nAgreement Matrix (row-normalized %):")
print(agreement_matrix.round(1))

# Reading this: each row sums to 100%
# A high diagonal value = strong agreement for that class
# 'Neutrals' typically has the lowest agreement — VADER struggles with neutral text
# 'Negatives' typically has the highest agreement — negative language is more distinct

In [ ]:
# Step 4: Per-class agreement rates — which sentiment class agrees most?
per_class_agreement = {}

for label in df['Sentiment'].unique():
    subset = df[df['Sentiment'] == label]
    rate = (subset['vader_provider_label'] == subset['Sentiment']).mean() * 100
    per_class_agreement[label] = round(rate, 1)

per_class_df = pd.Series(per_class_agreement).sort_values(ascending=False)

plt.figure(figsize=(8, 4))
per_class_df.plot(kind='bar', color=['lightgreen' if v >= 50 else 'lightsalmon' for v in per_class_df.values], edgecolor='grey')
plt.axhline(y=agreement_rate, color='steelblue', linestyle='--', label=f'Overall avg: {agreement_rate:.1f}%')
plt.title('VADER vs Provider Agreement Rate by Sentiment Class')
plt.xlabel('Provider Sentiment Label')
plt.ylabel('Agreement Rate (%)')
plt.xticks(rotation=0)
plt.legend()
plt.tight_layout()
plt.show()

print("\nPer-Class Agreement Rates:")
for label, rate in per_class_df.items():
    print(f"  {label}: {rate}%")

In [ ]:
# Step 5: Inspect the disagreements — most analytically interesting part
# These are posts where human labeling and automated VADER diverge
# In a healthcare context, understanding WHY they disagree is valuable

disagree_df = df[~agreement_mask][
    ['Sound Bite Text', 'Sentiment', 'vader_provider_label', 'vader_score']
].copy()

disagree_df = disagree_df.rename(columns={
    'Sentiment': 'Provider_Label',
    'vader_provider_label': 'VADER_Label'
})

print(f"Total disagreement cases: {len(disagree_df)} posts ({100 - agreement_rate:.1f}% of corpus)")
print("\nDisagreement breakdown — what VADER said when provider said X:")
print(pd.crosstab(disagree_df['Provider_Label'], disagree_df['VADER_Label']))

# The most interesting disagreements: Provider says Positive, VADER says Negative (or vice versa)
# These are posts with subtle sarcasm, medical caveats, or complex mixed language
hard_disagree = disagree_df[
    ((disagree_df['Provider_Label'] == 'Positives') & (disagree_df['VADER_Label'] == 'Negatives')) |
    ((disagree_df['Provider_Label'] == 'Negatives') & (disagree_df['VADER_Label'] == 'Positives'))
]

print(f"\nHard disagreements (Pos↔Neg flip): {len(hard_disagree)} posts")
print("\nSample hard disagreement posts:")
print("=" * 80)
for _, row in hard_disagree.sample(min(4, len(hard_disagree)), random_state=42).iterrows():
    print(f"Provider: {row['Provider_Label']} | VADER: {row['VADER_Label']} | Score: {row['vader_score']:.3f}")
    print(f"Text: {str(row['Sound Bite Text'])[:300]}")
    print("-" * 80)

In [ ]:
# Step 6: Summary interpretation for the report
print("=" * 60)
print("METHODOLOGICAL SUMMARY: VADER vs Provider Labels")
print("=" * 60)
print(f"""
Overall Agreement  : {agreement_rate:.1f}%
Best Agreement     : {per_class_df.index[0]} ({per_class_df.iloc[0]:.1f}%)
Worst Agreement    : {per_class_df.index[-1]} ({per_class_df.iloc[-1]:.1f}%)

Key Insight:
- 'Neutrals' agreement is lowest because VADER has no explicit
  neutral class — it defaults to near-zero scores which don't
  cleanly map to a provider's human-assigned Neutral label.

- 'Negatives' agreement is typically highest because negative
  language in CGM discussions (pain, failed, expensive, wrong)
  is lexically distinct and VADER's dictionary captures it well.

- Hard disagreements (Pos↔Neg flips) often contain medical
  caveats like 'no longer in pain' or sarcasm like
  'great, another sensor failure' — cases where sentence-level
  context matters more than individual word polarity.

Implication for Modeling:
  The ~{agreement_rate:.0f}% agreement rate means VADER and the provider
  are reasonably aligned but not identical. For downstream
  classification, we used the PROVIDER labels as ground truth
  since they reflect human judgment, while VADER scores serve
  as a continuous feature rather than a replacement label.
""")

## Aspect Based Sentiment Analysis

This section implements an Aspect-Based Sentiment Analysis pipeline to extract granular, feature-level feedback from unstructured text. By mapping predefined keywords to core product attributes (such as Sensor Life, Comfort, and Cost) and evaluating sentence-level sentiment using NLTK's VADER model, we transform raw user reviews into structured, quantifiable metrics. This allows us to move beyond generic document-level sentiment to pinpoint the exact drivers of user friction or satisfaction, ultimately providing targeted, actionable data to guide product iteration and strategic decision-making.

In [ ]:
# Aspect-Based Sentiment Analysis
# Instead of "this post is Positive", we get "accuracy = Negative, app = Positive"
# Much more useful for product teams than a single label

sid = SentimentIntensityAnalyzer()

aspects = {
    'Accuracy':     ['accurate', 'accuracy', 'reading', 'wrong', 'off', 'calibrat',
                     'fingerstick', 'false'],
    'Adhesive':     ['adhesive', 'stick', 'fall', 'fell', 'skin', 'rash', 'patch', 'itch'],
    'App & Alerts': ['app', 'alert', 'alarm', 'bluetooth', 'connect', 'sync', 'phone', 'notification'],
    'Cost':         ['insurance', 'cost', 'expensive', 'price', 'coverage', 'afford', 'copay'],
    'Comfort':      ['comfortable', 'pain', 'hurt', 'insertion', 'wear', 'site', 'arm', 'belly'],
    'Sensor Life':  ['sensor', 'failed', 'failure', 'replace', 'error', 'warmup', 'restart', 'faulty'],
}

def get_aspect_sentiment(text):
    if not isinstance(text, str):
        return {}

    result = {}
    words = text.lower().split()

    for aspect, keywords in aspects.items():
        # Find sentences containing aspect keywords
        sentences = text.split('.')
        aspect_sentences = [s for s in sentences if any(k in s.lower() for k in keywords)]

        if aspect_sentences:
            # Score the sentiment of those specific sentences
            scores = [sid.polarity_scores(s)['compound'] for s in aspect_sentences]
            result[aspect] = round(sum(scores) / len(scores), 3)
        else:
            result[aspect] = None  # aspect not mentioned in this post

    return result

df['aspect_sentiment'] = df['text_clean'].apply(get_aspect_sentiment)

# Expanding into separate columns for analysis
aspect_df = pd.json_normalize(df['aspect_sentiment'])
aspect_df.index = df.index

print("Aspect sentiment sample:")
print(aspect_df)

In [ ]:
# Comparing aspect-level sentiment between Dexcom and Freestyle Libre
# This is the most useful brand comparison you can make

dexcom_aspects = pd.json_normalize(df_dexcom['text_clean'].apply(get_aspect_sentiment)).mean()
libre_aspects  = pd.json_normalize(df_libre['text_clean'].apply(get_aspect_sentiment)).mean()

aspect_compare = pd.DataFrame({
    'Dexcom': dexcom_aspects,
    'Freestyle Libre': libre_aspects
}).dropna()

aspect_compare.plot(kind='bar', figsize=(12, 6), color=['lightskyblue', 'lightsalmon'], edgecolor='grey')

plt.title('Average Aspect-Level Sentiment: Dexcom vs Freestyle Libre')
plt.xlabel('Product Aspect')
plt.ylabel('Mean VADER Score (Higher = More Positive)')
plt.axhline(y=0, color='black', linestyle='--', linewidth=0.8)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print(aspect_compare.round(3))

# A bar dipping below 0 = patients are negative about that aspect for that brand
# This directly tells product teams WHERE to focus R&D

## TF - IDF differentiation

In [ ]:
# TF-IDF Differentiation — finding the vocabulary that is UNIQUE to each brand
# These are not just common words — they are words that appear proportionally
# more in one brand's posts than the other

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

# Fitting on combined corpus so the vocabulary is shared
combined = pd.concat([df_dexcom['lemmas_string'], df_libre['lemmas_string']])
tfidf.fit(combined)

dex_matrix  = tfidf.transform(df_dexcom['lemmas_string'])
libre_matrix = tfidf.transform(df_libre['lemmas_string'])

# Mean TF-IDF score per term for each brand
dex_scores  = pd.Series(dex_matrix.mean(axis=0).A1, index=tfidf.get_feature_names_out())
libre_scores = pd.Series(libre_matrix.mean(axis=0).A1, index=tfidf.get_feature_names_out())

# Differentiation score: how much higher is this term for Brand A vs Brand B
dex_signature  = (dex_scores - libre_scores).sort_values(ascending=False)
libre_signature = (libre_scores - dex_scores).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

dex_signature.head(15).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 15 Signature Terms — Dexcom')
axes[0].set_xlabel('Differential TF-IDF Score')
axes[0].invert_yaxis()

libre_signature.head(15).plot(kind='barh', ax=axes[1], color='salmon')
axes[1].set_title('Top 15 Signature Terms — Freestyle Libre')
axes[1].set_xlabel('Differential TF-IDF Score')
axes[1].invert_yaxis()

plt.suptitle('Brand Vocabulary Differentiation (What makes each brand\'s conversation unique)', fontsize=13)
plt.tight_layout()
plt.show()

## Brand Switch Detection

In [ ]:
# Detecting brand switching language — these posts are the most competitively valuable
# A patient switching from Dexcom to Libre (or vice versa) is a churn event

switch_keywords = ['switch', 'switched', 'switching', 'moved', 'moving', 'changed',
                   'upgrade', 'replace', 'replacing', 'went from', 'going from',
                   'used to', 'before i', 'previously']

def detect_switch(text):
    if not isinstance(text, str):
        return False
    text_lower = text.lower()
    return any(k in text_lower for k in switch_keywords)

df['is_switch_post'] = df['text_clean'].apply(detect_switch)

switchers = df[df['is_switch_post'] == True].copy()
print(f"Switch-related posts found: {len(switchers)} ({len(switchers)/len(df)*100:.1f}% of corpus)")

# Direction of switching
dex_to_libre = switchers[
    switchers['text_clean'].str.contains('dexcom', case=False) &
    switchers['text_clean'].str.contains('libre|freestyle', case=False)
]

# Sentiment of switchers — are people switching because they are unhappy?
print("\nSentiment of switch posts:")
print(switchers['Sentiment'].value_counts(normalize=True).round(3) * 100)

print(f"\nPosts mentioning BOTH brands (comparison/switch posts): {len(dex_to_libre)}")

# Average VADER score of switchers vs non-switchers
print(f"\nVADER score — Switchers: {switchers['vader_score'].mean():.4f}")
print(f"VADER score — Non-switchers: {df[~df['is_switch_post']]['vader_score'].mean():.4f}")

# Switcher posts are almost always more negative — this is your churn signal
plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x='is_switch_post', y='vader_score', palette='coolwarm')
plt.xticks([0, 1], ['Regular Posts', 'Switch-Related Posts'])
plt.title('Sentiment: Switch Posts vs Regular Posts')
plt.ylabel('VADER Compound Score')
plt.tight_layout()
plt.show()

## Unsupervised Models

### K-Means

In [ ]:
features = ['vader_score', 'Richness', 'Author Reddit Karma', 'Reddit Score','Source Type', 'Followers/Daily Unique Visitors/Subscribers']

In [ ]:
#These four features measure different dimensions: Emotional (Vader), Quantitative (Richness), Historical (Karma), and Social (Score). They don't "overlap," meaning each one adds fresh information to the distance calculation in K-Means.

In [ ]:
df_cluster = df[features].copy()

In [ ]:
# Social media metrics are power-law distributed. Log-scaling prevents
# 'Karma millionaires' from skewing the entire model.
df_cluster['Log_Karma'] = np.log1p(df_cluster['Author Reddit Karma'].clip(lower=0))
df_cluster['Log_Score'] = np.log1p(df_cluster['Reddit Score'].clip(lower=0))

In [ ]:
df_cluster = pd.get_dummies(df_cluster, columns=['Source Type'], drop_first=True)

In [ ]:
model_features = ['vader_score', 'Richness', 'Log_Karma', 'Log_Score', 'Followers/Daily Unique Visitors/Subscribers']

In [ ]:
scaler = StandardScaler()

In [ ]:
X_scaled = scaler.fit_transform(df_cluster[model_features])

#### Dexcom

In [ ]:
df_dexcom_cluster = df_dexcom[features].copy()

In [ ]:
df_dexcom_cluster['Log_Karma'] = np.log1p(df_dexcom_cluster['Author Reddit Karma'].clip(lower=0))
df_dexcom_cluster['Log_Score'] = np.log1p(df_dexcom_cluster['Reddit Score'].clip(lower=0))
df_dexcom_cluster['Log_Followers'] = np.log1p(df_dexcom_cluster['Followers/Daily Unique Visitors/Subscribers'].clip(lower=0))

In [ ]:
platform_dummies = pd.get_dummies(df_dexcom_cluster['Source Type'], prefix='Platform').astype(int)

In [ ]:
df_dexcom_cluster

In [ ]:
X_scaled = scaler.fit_transform(df_dexcom_cluster[model_features])

In [ ]:
X_final = np.hstack((X_scaled, platform_dummies.values))

In [ ]:
# Elbow Method to determine number of clusters

In [ ]:
# Range of clusters to evaluate
k_range = range(2, 11)
inertia = []
silhouette_avg = []

for k in k_range:
    # Initialize and fit KMeans
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_scaled)

    # WCSS for Elbow Method
    inertia.append(kmeans.inertia_)

    # Average Silhouette Score
    silhouette_avg.append(silhouette_score(X_scaled, cluster_labels))

In [ ]:
# Elbow Method
plt.plot(k_range, inertia, marker='o', linestyle='-', color='b')

plt.title('Elbow Method (Look for the "Bend")')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia (WCSS)')
plt.xticks(k_range)
plt.grid(True)


plt.show()

In [ ]:
# Solhouette Score
plt.plot(k_range, silhouette_avg, marker='o', linestyle='-', color='red')

plt.title('Silhouette Analysis (Higher is Better)')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Average Silhouette Score')
plt.xticks(k_range)
plt.grid(True)

plt.show()

In [ ]:
# Choosing 5 Clusters based on Elbow method

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df_dexcom_cluster['Customer_Segment'] = kmeans.fit_predict(X_final)

In [ ]:
df_dexcom_cluster['Customer_Segment'].value_counts().sort_index()

In [ ]:
# Calculating the mean of each feature per cluster
cluster_profile = df_dexcom_cluster.groupby('Customer_Segment')[model_features].mean()

# Add the count to the table for context
cluster_profile['count'] = df_dexcom_cluster['Customer_Segment'].value_counts()

# Displaying the table
display(cluster_profile.style.background_gradient(axis=0))

In [ ]:
# Heatmap.. Not that useful tbh

plt.figure(figsize=(12, 6))
sns.heatmap(cluster_profile.drop(columns=['count']), annot=True, cmap='viridis')
plt.title('Cluster Profiling: Feature Means per Segment')
plt.show()

The Segment *Personas*

Segment 0: The Vocal Critics

Traits: High engagement (Log_Karma) but the only group with a negative sentiment (vader_score of -0.47).

Comment: These are likely power users who are frustrated. They have a history in the community but are currently reporting bugs or venting about Dexcom issues.

Segment 1: The Contented Casuals

Traits: Positive sentiment but the lowest activity metrics across the board.

Comment: These users are happy but quiet. They represent a "low-maintenance" segment that likes the product but doesn't contribute much content.

Segment 2: The Loyalist Power Users

Traits: Highest engagement (Log_Karma / Log_Score) paired with very high positive sentiment.

Comment: This is your "Golden Segment." They are deeply integrated into the ecosystem and act as brand ambassadors.

Segment 3: The Mega-Influencers (The Outliers)

Traits: A tiny group (only 62 people) but with a massive average reach (2.8×10
7
  followers).

Comment: These aren't standard customers; they are likely high-profile influencers or media accounts. Their sentiment is positive, making them key for PR.

Segment 4: The Deep Researchers

Traits: Exceptionally high Richness score (4.4 vs. the ~1.4 average).

Comment: These users provide high-value, detailed content. They likely post long-form reviews, data-heavy analysis, or technical guides.

#### Libre

In [ ]:
df_Libre_cluster = df_libre[features].copy()

In [ ]:
df_Libre_cluster['Log_Karma'] = np.log1p(df_Libre_cluster['Author Reddit Karma'].clip(lower=0))
df_Libre_cluster['Log_Score'] = np.log1p(df_Libre_cluster['Reddit Score'].clip(lower=0))
df_Libre_cluster['Log_Followers'] = np.log1p(df_Libre_cluster['Followers/Daily Unique Visitors/Subscribers'].clip(lower=0))

In [ ]:
platform_dummies_2 = pd.get_dummies(df_Libre_cluster['Source Type'], prefix='Platform').astype(int)

In [ ]:
X_scaled = scaler.fit_transform(df_Libre_cluster[model_features])

In [ ]:
X_final = np.hstack((X_scaled, platform_dummies_2.values))

In [ ]:
# Range of clusters to evaluate
k_range = range(2, 11)
inertia = []
silhouette_avg = []

for k in k_range:
    # Initialize and fit KMeans
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_scaled)

    # WCSS for Elbow Method
    inertia.append(kmeans.inertia_)

    # Average Silhouette Score
    silhouette_avg.append(silhouette_score(X_scaled, cluster_labels))

In [ ]:
# Elbow Method
plt.plot(k_range, inertia, marker='o', linestyle='-', color='b')

plt.title('Elbow Method (Look for the "Bend")')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia (WCSS)')
plt.xticks(k_range)
plt.grid(True)


plt.show()

In [ ]:
# Solhouette Score
plt.plot(k_range, silhouette_avg, marker='o', linestyle='-', color='red')

plt.title('Silhouette Analysis (Higher is Better)')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Average Silhouette Score')
plt.xticks(k_range)
plt.grid(True)

plt.show()

In [ ]:
# Choosing 5 clusters due to elbow chart

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df_Libre_cluster['Customer_Segment'] = kmeans.fit_predict(X_final)

In [ ]:
df_Libre_cluster['Customer_Segment'].value_counts().sort_index()

In [ ]:
# Calculating the mean of each feature per cluster
cluster_profile = df_Libre_cluster.groupby('Customer_Segment')[model_features].mean()

# Add the count to the table for context
cluster_profile['count'] = df_Libre_cluster['Customer_Segment'].value_counts()

# Displaying the table
display(cluster_profile.style.background_gradient(axis=0))

In [ ]:
# Heatmap.. Not that useful tbh

plt.figure(figsize=(12, 6))
sns.heatmap(cluster_profile.drop(columns=['count']), annot=True, cmap='viridis')
plt.title('Cluster Profiling: Feature Means per Segment')
plt.show()

Segment 0: The Disgruntled Long-Termers

Traits: Negative sentiment (-0.46), high Log_Karma, and respectable Richness.

Comment: Similar to Dexcom's Segment 0, these are active community members who are currently unhappy. They aren't just "trolls"; they have the karma to prove they've been around, but their recent output is heavily critical.

Segment 1: The Technical Advocates

Traits: High positive sentiment (0.54) and the highest Richness score (4.43).

Comment: This is a high-value group. They aren't just happy; they are writing detailed, long-form content. These are likely the people writing "How-To" guides or detailed comparisons for Libre.

Segment 2: The Casual Observers

Traits: Very low activity (Log_Karma of 0.53) but generally positive sentiment.

Comment: Your largest group (1,301 users). These are "lurkers" who occasionally chime in with something positive but don't drive the conversation.

Segment 3: The Community Leaders

Traits: The highest Log_Karma and Log_Score among regular users, with high positive sentiment.

Comment: These are your true power users. They dominate the discussion and maintain a positive vibe. They are the backbone of the Libre online community.

Segment 4: The Outlier Influencers

Traits: A tiny group (41 users) with massive reach (2.9×10
7
  followers).

Comment: Just like in the Dexcom data, these are the "Super-Outliers." Interestingly, their sentiment is significantly lower (0.18) than the other positive groups, suggesting that when big accounts talk about Libre, they are more neutral or cautious than the "Technical Advocates."

## Brand-specific Feature Analysis — Praises and Complaints

In [ ]:
# Extracting what patients PRAISE about each brand (from positive posts)

dexcom_positive   = df_dexcom[df_dexcom['Sentiment'] == 'Positives']['text_clean']
libre_positive    = df_libre[df_libre['Sentiment'] == 'Positives']['text_clean']

# Top words in positive Dexcom posts
dex_pos_words = Counter(' '.join(dexcom_positive.dropna()).split())
lib_pos_words = Counter(' '.join(libre_positive.dropna()).split())

# Removing brand names themselves since they are obvious
for remove_word in ['dexcom', 'libre', 'freestyle', 'g', 'cgm']:
    dex_pos_words.pop(remove_word, None)
    lib_pos_words.pop(remove_word, None)

print('Top 30 words in POSITIVE Dexcom posts (what patients praise):')
for word, count in dex_pos_words.most_common(30):
    print(f'  {word}: {count}')

print()
print('Top 30 words in POSITIVE Freestyle Libre posts (what patients praise):')
for word, count in lib_pos_words.most_common(30):
    print(f'  {word}: {count}')

# An analysis of patient sentiment shows that Dexcom users prioritize ecosystem integration and reliability; praise heavily centers on the G6's accuracy, dependable alarms, and closed-loop capabilities with t:slim pumps. Conversely, Freestyle Libre users index highly on practical usability. Their praises focus on the device's physical comfort (smaller size, painless insertion) and everyday convenience (scan-free app, affordability, and ease of use)

In [ ]:
# Extracting what patients COMPLAIN about each brand (from negative posts)

dexcom_negative = df_dexcom[df_dexcom['Sentiment'] == 'Negatives']['text_clean']
libre_negative  = df_libre[df_libre['Sentiment'] == 'Negatives']['text_clean']

dex_neg_words = Counter(' '.join(dexcom_negative.dropna()).split())
lib_neg_words = Counter(' '.join(libre_negative.dropna()).split())

for remove_word in ['dexcom', 'libre', 'freestyle', 'g', 'cgm']:
    dex_neg_words.pop(remove_word, None)
    lib_neg_words.pop(remove_word, None)

print('Top 30 words in NEGATIVE Dexcom posts (what patients complain about):')
for word, count in dex_neg_words.most_common(30):
    print(f'  {word}: {count}')

print()
print('Top 30 words in NEGATIVE Freestyle Libre posts (what patients complain about):')
for word, count in lib_neg_words.most_common(30):
    print(f'  {word}: {count}')


In [ ]:
# Visual comparison of top complaint words — side by side horizontal bar

top_n = 12
dex_neg_top = dex_neg_words.most_common(top_n)
lib_neg_top = lib_neg_words.most_common(top_n)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh([w for w, _ in dex_neg_top][::-1], [c for _, c in dex_neg_top][::-1],
             color='lightskyblue', edgecolor='deepskyblue')
axes[0].set_title('Top Complaint Words — Dexcom')
axes[0].set_xlabel('Frequency')

axes[1].barh([w for w, _ in lib_neg_top][::-1], [c for _, c in lib_neg_top][::-1],
             color='lightsalmon', edgecolor='salmon')
axes[1].set_title('Top Complaint Words — Freestyle Libre')
axes[1].set_xlabel('Frequency')

plt.suptitle('Brand-Level Complaint Language Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Supervised Models

In [ ]:
# First we will be running the supervised Models on OverAll Data and then evaluate the models and run the best one on Brand Spefic Data to get more nuanced information

# Dropping rows with empty lemma strings to avoid model training issues
df_model = df[df['lemmas_string'].str.strip() != ''].copy()

print('Rows used for modeling:', len(df_model))

# Checking class distribution for the sentiment target variable

print(df_model['Sentiment'].value_counts().sort_index())

In [ ]:
Y = df_model['Sentiment']
X = df_model['lemmas_string']

In [ ]:
# 80/20 split with stratify to preserve class ratios

X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

print('Train size:', X_train.shape[0])
print('Test size :', X_test.shape[0])

In [ ]:
# TF-IDF vectorization — converts the lemmatized text into a numeric feature matrix

vectorizer       = TfidfVectorizer()


X_train_vec      = vectorizer.fit_transform(X_train)
X_test_vec       = vectorizer.transform(X_test)

### Logistic Regression

In [ ]:
# Since we have multiple labels in our Sentiment sooo we have made the class multinomial

Logistic_model = LogisticRegression( multi_class='multinomial',
                                    class_weight='balanced')

In [ ]:
Logistic_model.fit(X_train_vec,
                   y_train)

In [ ]:
# Testing Accuracy of the Model

y_pred   = Logistic_model.predict(X_test_vec)
lr_accuracy = accuracy_score(y_test, y_pred)

print('Test Accuracy:', round(lr_accuracy, 4))

In [ ]:
# Confusion Matrix for Logistic Regression
cm_lr = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=Logistic_model.classes_)
disp.plot(cmap='Blues', ax=plt.gca())
plt.title('Confusion Matrix — Logistic Regression')
plt.tight_layout()
plt.show()

# Most misclassifications are between adjacent classes (1&2, 3&4) which makes intuitive sense
# A post scored 2 by VADER vs 3 is only slightly different in language

In [ ]:
# Precision, Recall, F1 for each class
precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred)

classes = Logistic_model.classes_

for i, cls in enumerate(classes):
    print(f'Sentiment {cls}:')
    print(f'  Precision: {round(precision[i], 3)}')
    print(f'  Recall   : {round(recall[i], 3)}')
    print(f'  F1-Score : {round(f1[i], 3)}')
    print()



In [ ]:
# Top predictive lemmas per sentiment rating using model coefficients
coefs         = Logistic_model.coef_
feature_names = vectorizer.get_feature_names_out()
classes       = Logistic_model.classes_

for i, cls in enumerate(classes):
    top_indices = coefs[i].argsort()[-10:][::-1]
    top_words   = [feature_names[idx] for idx in top_indices]
    print(f'Sentiment {cls}: {top_words}')

# Sentiment Mixed shows words like love, expensive, inaccurate, convenient, - mixed signals
# Sentiment Negatives shows words like hate, painful, died, failed, - severely negative language
# Sentiment Neutrals shows words like thread, project, according, moving, - neutral language
# Sentiment Positives shows words like love, saved, recommend, useful, - highly positive language

In [ ]:
sentiment_categories = df_model['Sentiment'].unique()

for category in sentiment_categories:
    print(f'SENTIMENT {category.upper()} SAMPLES')

    # Filter the dataframe for the current sentiment category
    condition = df_model['Sentiment'] == category

    # Sample up to 3 posts (or fewer if the category has less than 3)
    samples = df_model[condition]['Sound Bite Text'].sample(
        n=min(3, condition.sum())
    )

    for text in samples:
        print(f'- {text[:500]}...')
        print()
    print()

### Random Forrest

In [ ]:
# Random Forest with max_depth tuned — tested 5, 10, 15, 20 and 15 gave best balance

rf_model = RandomForestClassifier(n_estimators=150,
                                  max_depth=15, random_state=42, class_weight='balanced_subsample')

In [ ]:
rf_model.fit(X_train_vec,
             y_train)

y_pred_rf = rf_model.predict(X_test_vec)
rf_acc = accuracy_score(y_test, y_pred_rf)

print(f'Random Forest Test Accuracy: {rf_acc:.4f}')
print()

In [ ]:
print('Classification Report:')
print(classification_report(y_test, y_pred_rf))

In [ ]:
# Confusion Matrix for Random Forest
cm_rf = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(8, 6))
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=rf_model.classes_)
disp_rf.plot(cmap='Oranges', ax=plt.gca())
plt.title('Confusion Matrix — Random Forest')
plt.tight_layout()
plt.show()

#### Comparison of two models

In [ ]:
# Comparing both models side by side

print('Model Comparison')
print(f'Logistic Regression Accuracy: {lr_accuracy:.4f}')
print(f'Random Forest Accuracy       : {rf_acc:.4f}')
print()

precision_lr, recall_lr, f1_lr, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')
precision_rf, recall_rf, f1_rf, _ = precision_recall_fscore_support(y_test, y_pred_rf, average='weighted')

comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Weighted Precision', 'Weighted Recall', 'Weighted F1'],
    'Logistic Regression': [lr_accuracy, precision_lr, recall_lr, f1_lr],
    'Random Forest':       [rf_acc, precision_rf, recall_rf, f1_rf]
}).round(4)

print(comparison_df.to_string(index=False))


#### Applying Logistic Regression to Brand Specific Data

In [ ]:
# Applying best model to brand-specific data for targeted sentiment analysis
# This tells us the model-predicted sentiment breakdown for each brand

df_dexcom_model = df_dexcom[df_dexcom['lemmas_string'].str.strip() != ''].copy()

df_libre_model  = df_libre[df_libre['lemmas_string'].str.strip() != ''].copy()

In [ ]:
# Vectorizing the data
dexcom_vec    = vectorizer.transform(df_dexcom_model['lemmas_string'])
libre_vec     = vectorizer.transform(df_libre_model['lemmas_string'])

In [ ]:
df_dexcom_model['pred_sentiment'] = Logistic_model.predict(dexcom_vec)

df_libre_model['pred_sentiment']  = Logistic_model.predict(libre_vec)

In [ ]:
print("DEXCOM: MODEL PERFORMANCE")
print(classification_report(df_dexcom_model['Sentiment'], df_dexcom_model['pred_sentiment']))

print()

print("FREESTYLE LIBRE: MODEL PERFORMANCE")
print(classification_report(df_libre_model['Sentiment'], df_libre_model['pred_sentiment']))

# Extracting weighted metrics for side-by-side comparison

# Dexcom metrics
dex_acc = accuracy_score(df_dexcom_model['Sentiment'], df_dexcom_model['pred_sentiment'])
dex_prec, dex_rec, dex_f1, _ = precision_recall_fscore_support(
    df_dexcom_model['Sentiment'], df_dexcom_model['pred_sentiment'], average='weighted'
)

# Freestyle Libre metrics
lib_acc = accuracy_score(df_libre_model['Sentiment'], df_libre_model['pred_sentiment'])
lib_prec, lib_rec, lib_f1, _ = precision_recall_fscore_support(
    df_libre_model['Sentiment'], df_libre_model['pred_sentiment'], average='weighted'
)

# Building and display the comparison DataFrame
brand_comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Weighted Precision', 'Weighted Recall', 'Weighted F1'],
    'Dexcom': [dex_acc, dex_prec, dex_rec, dex_f1],
    'Freestyle Libre': [lib_acc, lib_prec, lib_rec, lib_f1]
}).round(4)

print("BRAND PERFORMANCE COMPARISON")

print(brand_comparison_df.to_string(index=False))

Key Findings from the Brand Performance Comparison:
1. Unbiased Performance: The Weighted F1-scores across Dexcom (0.63) and
   Freestyle Libre (0.65) are nearly identical. This confirms the model understands
   the colloquial vocabulary of both user bases equally well.
2. High Negative Recall for Dexcom: The model achieved an impressive 83% recall
   on Dexcom 'Negatives'. This indicates that Dexcom users express dissatisfaction
   using highly distinct and easily classifiable language compared to Libre users.
3. Business Validity: Because the model is unbiased, the previously observed
   spike in negative sentiment for Dexcom (20.5% vs Libre's 11.8%) is mathematically
   sound and represents a genuine operational risk for the Dexcom brand.

In [ ]:
dex_pred_dist  = df_dexcom_model['pred_sentiment'].value_counts(normalize=True).sort_index() * 100
lib_pred_dist  = df_libre_model['pred_sentiment'].value_counts(normalize=True).sort_index() * 100

pred_comp = pd.DataFrame({'Dexcom': dex_pred_dist, 'Freestyle Libre': lib_pred_dist}).fillna(0)

pred_comp.plot(kind='bar', figsize=(10, 6), color=['lightskyblue', 'lightsalmon'],
               edgecolor='grey')
plt.xlabel('Predicted Sentiment Rating (Mixed, Negative Neutral and Positive)')
plt.ylabel('Percentage of Posts (%)')
plt.title('Model-Predicted Sentiment Distribution by Brand')
plt.xticks(rotation=0)
plt.legend()
plt.tight_layout()
plt.show()

print(pred_comp.round(2))


## K-Fold Cross Validation

In [ ]:
# K-Fold Cross-Validation for robust model evaluation
# A single train/test split can be lucky or unlucky depending on the random state
# K-Fold gives us a distribution of accuracy scores across 5 different splits
# This is the standard way to evaluate a model's true generalization ability

# Rebuilding the full feature matrix for cross-validation
# We use the same TF-IDF vectorizer fitted earlier
X_all = vectorizer.transform(df_model['lemmas_string'])
y_all = df_model['Sentiment']

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Logistic Regression K-Fold
lr_cv_scores = cross_val_score(Logistic_model, X_all, y_all, cv=kf, scoring='accuracy')

# Random Forest K-Fold
rf_cv_scores = cross_val_score(rf_model, X_all, y_all, cv=kf, scoring='accuracy')

print("Logistic Regression — 5-Fold Cross-Validation:")
print(f"  Fold Scores : {[round(s, 4) for s in lr_cv_scores]}")
print(f"  Mean Accuracy : {lr_cv_scores.mean():.4f}")
print(f"  Std Dev       : {lr_cv_scores.std():.4f}")
print()
print("Random Forest — 5-Fold Cross-Validation:")
print(f"  Fold Scores : {[round(s, 4) for s in rf_cv_scores]}")
print(f"  Mean Accuracy : {rf_cv_scores.mean():.4f}")
print(f"  Std Dev       : {rf_cv_scores.std():.4f}")

# The std dev tells us how stable the model is — lower is better
# A low std dev means the model is consistent across different subsets of data
# This is more trustworthy than a single test-set accuracy number

In [ ]:
cv_df = pd.DataFrame({
    'Logistic Regression': lr_cv_scores,
    'Random Forest': rf_cv_scores
})

cv_df.plot(kind='box', figsize=(8, 5))
plt.title('5-Fold Cross-Validation Accuracy Distribution by Model')
plt.ylabel('Accuracy')
plt.tight_layout()
plt.show()

#Phase 5: Business Insights & Strategic Recommendations

In [ ]:
# Setting up the Gemini API key and model
API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=API_KEY)

gemini_model = genai.GenerativeModel('gemini-2.5-flash')

def get_completion(prompt):
    generation_config = {'temperature': 0}
    response = gemini_model.generate_content(prompt, generation_config=generation_config)
    return response.text

In [ ]:
# Setting up CHATGPT


API_KEY = userdata.get('OPENAI_API_KEY')
BASE_URL = "https://ai-120oss.devplusops.com/v1"

client_1 = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)

# 2. Functional wrapper for completions
def get_chat_completion(prompt, model="gpt-oss-120b"):
    response = client_1.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0  # Deterministic output
    )
    return response.choices[0].message.content

In [ ]:
# Setting up Groq

api_key = userdata.get('GROQ_API_KEY').strip()
client_2 = Groq(api_key=api_key)

def get_groq_completion(prompt):
    completion = client_2.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )
    return completion.choices[0].message.content

## Overall Landscape

### Prompt

In [ ]:
# Sampling high-engagement posts from each topic cluster for the LLM
# Using posts with highest engagement as they represent the strongest community signals

top_engaged = df.nlargest(21, 'Reddit Score')[['Sound Bite Text', 'Sentiment', 'Source Name']]

formatted_posts = ''
for i, row in top_engaged.iterrows():
    formatted_posts += f'{i+1}. [{row["Sentiment"]}] {str(row["Sound Bite Text"])[:400]}\n\n'

persona    = '**Persona:** Act as a senior data scientist and healthcare market research analyst with deep expertise in medical device consumer behavior.'
audience   = '**Audience:** You are presenting to the product strategy board of a major CGM manufacturer.'
ask        = ('**Ask:** Based on the 5 highest-engagement social media posts about continuous glucose monitoring below, '
              'identify: (1) The 5 most critical patient needs, (2) The 3 biggest market opportunities, '
              '(3) The most significant emotional pain points patients express.')
context    = f'**Context (Top-Engaged CGM Social Media Posts):**\n{formatted_posts}'
boundaries = '**Boundaries:** Be specific and data-driven. Use bullet points. Keep response under 500 words. Do not make up statistics.'

prompt_51 = '\n\n'.join([persona, audience, ask, context, boundaries])

display(Markdown(prompt_51))

### Chatgpt Response

In [ ]:
response_chat = get_chat_completion(prompt_51)
display(Markdown(response_chat))

### Groq Response

In [ ]:
response_51 = get_groq_completion(prompt_51)
display(Markdown(response_51))

## Product Recommendation

### Prompt

In [ ]:
# Gathering quantitative evidence to support the recommendation
# We want to combine: average VADER score, % positive posts, share of voice, complaint severity

dexcom_avg_vader = df_dexcom['vader_score'].mean()
libre_avg_vader  = df_libre['vader_score'].mean()

dexcom_pct_pos = (df_dexcom['Sentiment'] == 'Positives').mean() * 100
libre_pct_pos  = (df_libre['Sentiment'] == 'Positives').mean() * 100

dexcom_pct_neg = (df_dexcom['Sentiment'] == 'Negatives').mean() * 100
libre_pct_neg  = (df_libre['Sentiment'] == 'Negatives').mean() * 100

dexcom_post_count = len(df_dexcom)
libre_post_count  = len(df_libre)

quant_summary = f'''
Quantitative Data for Recommendation:

Dexcom:
  - Total Posts (Share of Voice): {dexcom_post_count}
  - Average VADER Score: {dexcom_avg_vader:.4f}
  - % Positive Posts: {dexcom_pct_pos:.1f}%
  - % Negative Posts: {dexcom_pct_neg:.1f}%

Freestyle Libre:
  - Total Posts (Share of Voice): {libre_post_count}
  - Average VADER Score: {libre_avg_vader:.4f}
  - % Positive Posts: {libre_pct_pos:.1f}%
  - % Negative Posts: {libre_pct_neg:.1f}%
'''

print(quant_summary)

In [ ]:
persona_52  = '**Persona:** Act as a senior healthcare data scientist who has just completed a comprehensive NLP analysis of 37,000+ CGM social media posts.'
audience_52 = '**Audience:** A diabetes patient who is newly diagnosed with Type 1 diabetes and is deciding between Dexcom and Freestyle Libre as their first CGM.'
ask_52 = ('**Ask:** Based on the quantitative NLP analysis results and social media insights below, provide:\n'
          '1. A clear product recommendation (Dexcom or Freestyle Libre) with reasoning\n'
          '2. Specific use cases where each product excels (e.g., athletes, newly diagnosed, budget-conscious, tech-savvy)\n'
          '3. What the patient should watch out for with the recommended product\n'
          '4. One key feature gap the recommended brand should fill based on patient feedback')
context_52  = f'**Context (Analysis Results):**\n{quant_summary}'
boundaries_52 = '**Boundaries:** Be decisive and patient-centric. Ground recommendation in the data. Keep response under 400 words. No medical advice — focus on product/experience factors.'

prompt_52 = '\n\n'.join([persona_52, audience_52, ask_52, context_52, boundaries_52])


display(Markdown(prompt_52))

### Chat Response

In [ ]:
response_chat_2 = get_chat_completion(prompt_52)
display(Markdown(response_chat_2))

### Groq Response


In [ ]:
response_52 = get_groq_completion(prompt_52)
display(Markdown(response_52))

## Brand Strategy

### Prompt

In [ ]:
# Pulling the top unmet needs and complaint categories into a structured format for Claude

# Reusing unmet_df from Phase 3
unmet_str = unmet_df.to_string(index=False)

# Top negative words for each brand (extracted earlier)
dex_neg_summary = ', '.join([w for w, _ in dex_neg_words.most_common(15)])
lib_neg_summary = ', '.join([w for w, _ in lib_neg_words.most_common(15)])

persona_53  = '**Persona:** Act as a product strategy consultant specializing in digital health and medical wearables.'
audience_53 = '**Audience:** The C-suite product and marketing leadership teams at Dexcom and Abbott (Freestyle Libre).'
ask_53 = ('**Ask:** Based on the social media NLP analysis data below, provide a concrete brand strategy for EACH company:\n'
          '1. Dexcom: Top 3 specific product improvements to prioritize based on patient complaints\n'
          '2. Dexcom: One marketing message that would resonate most with the online patient community\n'
          '3. Freestyle Libre: Top 3 specific product improvements to prioritize based on patient complaints\n'
          '4. Freestyle Libre: One marketing message that would resonate most with the online patient community\n'
          '5. One emerging use case neither brand is currently marketing that NLP data suggests has strong demand')
context_53  = (
    f'**Context (NLP Analysis Results):**\n\n'
    f'Unmet Need Categories (post count):\n{unmet_str}\n\n'
    f'Top Complaint Words for Dexcom: {dex_neg_summary}\n'
    f'Top Complaint Words for Freestyle Libre: {lib_neg_summary}\n'
)
boundaries_53 = '**Boundaries:** Be specific and actionable — not generic advice. Tie every recommendation back to the data. Bullet points only. Under 600 words total.'

prompt_53 = '\n\n'.join([persona_53, audience_53, ask_53, context_53, boundaries_53])

display(Markdown(prompt_53))

### Chat response

In [ ]:
response_chat_3 = get_chat_completion(prompt_53)
display(Markdown(response_chat_3))

### Groq response

In [ ]:
response_53 = get_groq_completion(prompt_53)
display(Markdown(response_53))

## Use cases

In [ ]:
# 1. Actual VADER sentiment comparison
vader_summary = (
    "Dexcom: Mean VADER compound score = 0.2219\n"
    "Freestyle Libre: Mean VADER compound score = 0.3285\n"
    "Insight: Libre holds a noticeably higher average sentiment and a larger proportion of 'Very Positive' (5) scores relative to its total volume compared to Dexcom."
)

# 2. Actual K-Means Clusters (Personas derived from cluster profiles)
kmeans_summary = """
Dexcom Segments:
- Seg 0 (Vocal Critics): High engagement, negative sentiment (-0.47). Frustrated power users reporting bugs.
- Seg 1 (Contented Casuals): Positive sentiment, lowest activity. Happy but quiet.
- Seg 2 (Loyalist Power Users): Highest engagement and very high positive sentiment. Golden brand ambassadors.
- Seg 3 (Mega-Influencers): Tiny group (62 users), massive reach (28M followers), positive sentiment.
- Seg 4 (Deep Researchers): Exceptionally high Richness (4.4). Provide long-form reviews and technical analysis.

Libre Segments:
- Seg 0 (Disgruntled Long-Termers): Negative sentiment (-0.46), high karma. Active but highly critical.
- Seg 1 (Technical Advocates): High positive sentiment (0.54), highest Richness (4.43). Write detailed 'How-To' guides.
- Seg 2 (Casual Observers): Largest group (1,301 users), very low activity, generally positive.
- Seg 3 (Community Leaders): Highest engagement among regular users, high positive sentiment. Backbone of community.
- Seg 4 (Outlier Influencers): Tiny group (41 users), massive reach (29M followers), but sentiment is notably lower/cautious (0.18) compared to other positive groups.
"""

# 3. Model Performance Reality Check
model_performance_summary = (
    "Predictive Modeling (Classification): \n"
    "Logistic Regression Accuracy: 0.5237 (Weighted F1: 0.5574)\n"
    "Random Forest Accuracy: 0.5212 (Weighted F1: 0.5488)\n"
    "Insight: Our predictive models are performing at ~52% accuracy, which is practically a coin toss."
)


persona_55    = '**Persona:** Act as a Principal Data Scientist and Lead Product Strategist.'
audience_55   = '**Audience:** You are presenting to the Executive Board of Freestyle Libre (the winning brand by overall sentiment).'
ask_55        = ('**Ask:** Based on the machine learning outputs provided (K-Means segments, VADER sentiment comparison, '
              'and our current predictive model limitations), formulate ONE or TWO highly specific, actionable business use cases. '
              'Explain exactly how the brand can operationalize these segment insights in a real-world scenario.')
context_55    = (f'**Context (ML Model Outputs):**\n'
              f'1. Sentiment Split:\n{vader_summary}\n\n'
              f'2. Key Customer Segments (K-Means):\n{kmeans_summary}\n\n'
              f'3. Model Performance Constraints:\n{model_performance_summary}')
boundaries_55 = ('**Boundaries:** Be brutally honest and highly pragmatic. Focus ONLY on 1 or 2 specific use cases. '
              'Avoid generic marketing fluff. Ground the operationalization entirely in the provided data. '
              'Crucially: You MUST address how to safely utilize these insights given the 52% predictive model accuracy '
              '(e.g., recommend broad community-level strategies or deterministic rule-based interventions based on cluster profiles, '
              'rather than automated, individualized ML predictions). Use bullet points and keep the response under 400 words.')

prompt_55 = '\n\n'.join([persona_55, audience_55, ask_55, context_55, boundaries_55])

display(Markdown(prompt_55))

### Chat Response

In [ ]:
response_chat_5 = get_chat_completion(prompt_55)
display(Markdown(response_chat_5))

### Groq Response

In [ ]:
response_groq_55 = get_groq_completion(prompt_55)

display(Markdown(response_groq_55))

## Actionable Steps

### Prompt

In [ ]:
dexcom_avg_vader = df_dexcom['vader_score'].mean()
libre_avg_vader  = df_libre['vader_score'].mean()

dexcom_pct_pos = (df_dexcom['Sentiment'] == 'Positives').mean() * 100
libre_pct_pos  = (df_libre['Sentiment'] == 'Positives').mean() * 100

dexcom_pct_neg = (df_dexcom['Sentiment'] == 'Negatives').mean() * 100
libre_pct_neg  = (df_libre['Sentiment'] == 'Negatives').mean() * 100

In [ ]:
# Gathering the key outputs from all phases into a single summary for the LLM
# This gives the model everything it needs to generate concrete, data-grounded next steps

# Model performance summary
model_eval_summary = (
    f"Logistic Regression: Mean CV Accuracy = {lr_cv_scores.mean():.4f} (Std: {lr_cv_scores.std():.4f})\n"
    f"Random Forest: Mean CV Accuracy = {rf_cv_scores.mean():.4f} (Std: {rf_cv_scores.std():.4f})\n"
    f"Key Insight: Both models perform at ~52-54% accuracy. "
    f"The four-class imbalance (Positives/Negatives/Neutrals/Mixed) is the main driver of this ceiling. "
    f"Rule-based interventions using K-Means segments are more reliable than automated ML predictions."
)

# Top unmet needs (from Phase 3)
top_needs = unmet_df.head(4)['Unmet Need Category'].tolist()
top_needs_str = ', '.join(top_needs)

# Brand sentiment gap (from Phase 4)
brand_gap_summary = (
    f"Dexcom: {dexcom_pct_pos:.1f}% positive, {dexcom_pct_neg:.1f}% negative, "
    f"Mean VADER = {dexcom_avg_vader:.4f}\n"
    f"Freestyle Libre: {libre_pct_pos:.1f}% positive, {libre_pct_neg:.1f}% negative, "
    f"Mean VADER = {libre_avg_vader:.4f}"
)

# K-Means segment summary (reusing from Phase 4)
segment_summary = (
    "Dexcom has a 'Vocal Critics' segment (Segment 0) with negative VADER (-0.47) and high engagement. "
    "Libre's 'Disgruntled Long-Termers' (Segment 0) share similar traits. "
    "Both brands have high-reach Mega-Influencer segments (62 and 41 users) with positive sentiment. "
    "Freestyle Libre's 'Technical Advocates' (Segment 1) are writing detailed how-to content — a retention asset."
)

print("Context assembled. Running Actionable Steps prompt...")

In [ ]:
# Constructing the Actionable Steps prompt
# This is the final deliverable of the project — turning all NLP outputs into a
# prioritized action plan that a real CGM company could act on tomorrow

persona_act  = (
    "**Persona:** Act as a Chief Data Officer presenting the final findings of a 6-week "
    "NLP research engagement to the executive leadership team of a CGM manufacturer."
)

audience_act = (
    "**Audience:** C-suite executives (CEO, CMO, Chief Medical Officer) at a CGM company "
    "who need a clear, prioritized action plan — not academic analysis."
)

ask_act = (
    "**Ask:** Based on the complete NLP analysis of 37,000+ CGM social media posts, "
    "provide a prioritized list of FIVE actionable next steps. For each step:\n"
    "1. State the specific action in one sentence\n"
    "2. Name the team responsible (Product, Marketing, Data Science, Medical Affairs)\n"
    "3. Cite the specific data finding that justifies this action\n"
    "4. Estimate the timeline (Quick Win = <3 months, Medium = 3-6 months, Strategic = 6-12 months)"
)

context_act = (
    f"**Context (Full Analysis Summary):**\n\n"
    f"Top 4 Unmet Patient Needs (by post volume): {top_needs_str}\n\n"
    f"Brand Sentiment Gap:\n{brand_gap_summary}\n\n"
    f"Customer Segment Intelligence:\n{segment_summary}\n\n"
    f"Predictive Model Constraints:\n{model_eval_summary}"
)

boundaries_act = (
    "**Boundaries:** Every recommendation must map to a specific data finding from above. "
    "No generic advice. Be ruthlessly specific — name features, campaigns, or data pipelines. "
    "Format: numbered list with sub-bullets for Team, Data Justification, and Timeline. "
    "Keep total response under 500 words."
)

prompt_act = '\n\n'.join([persona_act, audience_act, ask_act, context_act, boundaries_act])

display(Markdown(prompt_act))

### Groq Response

In [ ]:
# Groq Response — Actionable Steps
response_groq_act = get_groq_completion(prompt_act)
display(Markdown(response_groq_act))

### ChatGPT Response

In [ ]:
# ChatGPT Response — Actionable Steps
response_chat_act = get_chat_completion(prompt_act)
display(Markdown(response_chat_act))

### Gemini Response

In [ ]:
response_gemini = get_completion(prompt_act)
display(Markdown(response_gemini))